<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Homomorphic_encryption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# HOMOMORPHIC ENCRYPTION FOR SECURE FEDERATED AGGREGATION
# =========================

from google.colab import drive
drive.mount('/content/drive')

# =========================
# INSTALL LIBRARY
# =========================
!pip install tenseal -q

# =========================
# IMPORT LIBRARIES
# =========================
import tenseal as ts
import numpy as np
import pickle
import time

print("=" * 65)
print(" HOMOMORPHIC ENCRYPTION FOR SECURE FEDERATED AGGREGATION ")
print("=" * 65)
print("TenSEAL version:", ts.__version__)
print("All libraries imported successfully!")

# =========================
# LOAD FEDERATED MODELS
# =========================
print("\nLoading federated models from Google Drive...")

with open('/content/drive/MyDrive/fraud_detection_project/federated_model.pkl', 'rb') as f:
    bank_models = pickle.load(f)

print("Federated models loaded successfully!")
print(f"Number of local bank models: {len(bank_models)}")
print(f"Model type: {type(bank_models[0])}")

# =========================
# CREATE ENCRYPTION CONTEXT
# =========================
print("\nCreating Homomorphic Encryption context...")

context = ts.context(
    ts.SCHEME_TYPE.CKKS,              # Best for decimal values
    poly_modulus_degree=8192,         # Security parameter
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)

context.generate_galois_keys()
context.global_scale = 2**40

print("Encryption context created successfully!")
print("Scheme         : CKKS")
print("Security level : 128-bit")
print("Public key     : Ready")
print("Private key    : Ready")

# =========================
# EXTRACT LOCAL MODEL PARAMETERS
# =========================
print("\nExtracting local model feature importance vectors...")

bank_a_importance = bank_models[0].feature_importances_.tolist()
bank_b_importance = bank_models[1].feature_importances_.tolist()
bank_c_importance = bank_models[2].feature_importances_.tolist()

print(f"Number of features encrypted per bank: {len(bank_a_importance)}")

print("\nSample original feature importance values (Bank A):")
print(np.round(bank_a_importance[:10], 6))

# =========================
# ENCRYPT MODEL PARAMETERS
# =========================
print("\nEncrypting local model parameters...")
print("=" * 65)

start_encrypt = time.time()

encrypted_a = ts.ckks_vector(context, bank_a_importance)
encrypted_b = ts.ckks_vector(context, bank_b_importance)
encrypted_c = ts.ckks_vector(context, bank_c_importance)

encrypt_time = time.time() - start_encrypt

print("Bank A parameters encrypted successfully ✅")
print("Bank B parameters encrypted successfully ✅")
print("Bank C parameters encrypted successfully ✅")

print(f"\nEncryption time: {round(encrypt_time, 2)} seconds")
print("\nEncrypted vector preview (Bank A):")
print(str(encrypted_a)[:120], "...")

print("\nOriginal parameters  → readable numeric values")
print("Encrypted parameters → unreadable ciphertext")
print("Server cannot see actual model parameters!")

# =========================
# PRIVACY DEMONSTRATION
# =========================
print("\n" + "=" * 65)
print(" PRIVACY DEMONSTRATION ")
print("=" * 65)

print("\nWhat a BANK can see (original local parameters):")
print(f"feature_importance[0] = {round(bank_a_importance[0], 6)}")
print(f"feature_importance[1] = {round(bank_a_importance[1], 6)}")
print(f"feature_importance[2] = {round(bank_a_importance[2], 6)}")

print("\nWhat the SERVER can see (encrypted form only):")
print(str(encrypted_a)[:150], "...")

print("\nConclusion:")
print("The server cannot understand the original parameter values.")
print("Only the owner with the private key can decrypt them.")

# =========================
# HOMOMORPHIC AGGREGATION
# =========================
print("\n" + "=" * 65)
print(" HOMOMORPHIC AGGREGATION ON ENCRYPTED PARAMETERS ")
print("=" * 65)

print("\nServer aggregating encrypted parameters from all 3 banks...")

start_agg = time.time()

# Secure aggregation without decryption
aggregated_encrypted = encrypted_a + encrypted_b + encrypted_c

aggregation_time = time.time() - start_agg

print("Encrypted aggregation completed successfully ✅")
print(f"Aggregation time: {round(aggregation_time, 4)} seconds")

print("\nImportant:")
print("Aggregation happened directly on encrypted data.")
print("The server never decrypted or viewed any raw parameter values.")

# =========================
# DECRYPT AGGREGATED RESULT
# =========================
print("\nDecrypting aggregated result...")

start_decrypt = time.time()

decrypted_result = aggregated_encrypted.decrypt()
final_importance = np.array(decrypted_result) / 3

decrypt_time = time.time() - start_decrypt

print(f"Decryption time: {round(decrypt_time, 2)} seconds")

# =========================
# VERIFY CORRECTNESS
# =========================
print("\n" + "=" * 65)
print(" VERIFICATION OF HOMOMORPHIC ENCRYPTION CORRECTNESS ")
print("=" * 65)

# Plain average for correctness check
plain_average = (
    np.array(bank_a_importance) +
    np.array(bank_b_importance) +
    np.array(bank_c_importance)
) / 3

difference = np.abs(plain_average - final_importance)
max_diff = np.max(difference)
mean_diff = np.mean(difference)

print("Comparing plain aggregation vs encrypted aggregation...\n")

for i in range(5):
    print(f"Feature {i}:")
    print(f"  Plain Average      : {round(plain_average[i], 6)}")
    print(f"  Decrypted Average  : {round(final_importance[i], 6)}")
    print(f"  Difference         : {round(abs(plain_average[i] - final_importance[i]), 10)}")
    print()

print(f"Maximum difference across all features : {round(max_diff, 10)}")
print(f"Average difference across all features : {round(mean_diff, 10)}")

print("\nConclusion:")
print("Encrypted aggregation gives nearly the same result as plain aggregation.")
print("Tiny differences are expected due to floating-point approximation in CKKS.")
print("Homomorphic Encryption works correctly while preserving privacy!")

# =========================
# SAVE ENCRYPTED PARAMETERS
# =========================
print("\nSaving encrypted outputs to Google Drive...")

encrypted_a_bytes = encrypted_a.serialize()
encrypted_b_bytes = encrypted_b.serialize()
encrypted_c_bytes = encrypted_c.serialize()

with open('/content/drive/MyDrive/fraud_detection_project/encrypted_bank_a.pkl', 'wb') as f:
    pickle.dump(encrypted_a_bytes, f)

with open('/content/drive/MyDrive/fraud_detection_project/encrypted_bank_b.pkl', 'wb') as f:
    pickle.dump(encrypted_b_bytes, f)

with open('/content/drive/MyDrive/fraud_detection_project/encrypted_bank_c.pkl', 'wb') as f:
    pickle.dump(encrypted_c_bytes, f)

# Save encryption context (including secret key for demo)
context_bytes = context.serialize(save_secret_key=True)
with open('/content/drive/MyDrive/fraud_detection_project/he_context.pkl', 'wb') as f:
    pickle.dump(context_bytes, f)

print("encrypted_bank_a.pkl saved ✅")
print("encrypted_bank_b.pkl saved ✅")
print("encrypted_bank_c.pkl saved ✅")
print("he_context.pkl saved ✅")

# =========================
# SAVE RESULTS SUMMARY
# =========================
results_text = f"""
HOMOMORPHIC ENCRYPTION RESULTS
==============================
Library Used         : TenSEAL {ts.__version__}
Scheme               : CKKS
Security Level       : 128-bit
Encrypted Parameters : {len(bank_a_importance)} feature importance values

Timing:
Encryption Time      : {round(encrypt_time, 2)} seconds
Aggregation Time     : {round(aggregation_time, 4)} seconds
Decryption Time      : {round(decrypt_time, 2)} seconds

Verification:
Maximum Difference   : {round(max_diff, 10)}
Average Difference   : {round(mean_diff, 10)}

Privacy Achieved:
Level 1 - Federated Learning
Raw transaction data never leaves the bank

Level 2 - Homomorphic Encryption
Even model parameters are encrypted before sharing

Conclusion:
Encrypted aggregation successfully preserves privacy
while producing nearly identical results to plain aggregation.
"""

with open('/content/drive/MyDrive/fraud_detection_project/homomorphic_encryption_results.txt', 'w') as f:
    f.write(results_text)

print("homomorphic_encryption_results.txt saved ✅")

# =========================
# FINAL SUMMARY
# =========================
print("\n" + "=" * 65)
print(" FINAL HOMOMORPHIC ENCRYPTION SUMMARY ")
print("=" * 65)

print(f"Library used              : TenSEAL {ts.__version__}")
print(f"Encryption scheme         : CKKS")
print(f"Security level            : 128-bit")
print(f"Parameters encrypted      : {len(bank_a_importance)}")
print(f"Encryption time           : {round(encrypt_time, 2)} seconds")
print(f"Encrypted aggregation time: {round(aggregation_time, 4)} seconds")
print(f"Decryption time           : {round(decrypt_time, 2)} seconds")
print(f"Max difference            : {round(max_diff, 10)}")
print(f"Mean difference           : {round(mean_diff, 10)}")

print("\nPrivacy Levels Achieved:")
print("1. Federated Learning")
print("   → Raw transaction data never leaves the bank")
print("2. Homomorphic Encryption")
print("   → Even model parameters remain encrypted")
print("   → Server cannot read the model updates")

print("\nFinal Conclusion:")
print("A secure and privacy-preserving fraud detection framework")
print("was successfully implemented using Federated Learning")
print("combined with Homomorphic Encryption.")
print("=" * 65)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 HOMOMORPHIC ENCRYPTION FOR SECURE FEDERATED AGGREGATION 
TenSEAL version: 0.3.16
All libraries imported successfully!

Loading federated models from Google Drive...
Federated models loaded successfully!
Number of local bank models: 3
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>

Creating Homomorphic Encryption context...
Encryption context created successfully!
Scheme         : CKKS
Security level : 128-bit
Public key     : Ready
Private key    : Ready

Extracting local model feature importance vectors...
Number of features encrypted per bank: 224

Sample original feature importance values (Bank A):
[0.003344 0.004541 0.04501  0.000923 0.001856 0.013247 0.000242 0.000956
 0.011867 0.000597]

Encrypting local model parameters...
Bank A parameters encrypted successfully ✅
Bank B parameters encrypted successfully ✅
Bank C parameters enc